# Prompt Caching: Paying Once for a Prefix

The runnable code of [Prompt Caching: Paying Once for a Prefix](https://residual-stream.lvtan71.workers.dev/posts/prompt-caching/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/prompt-caching.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 3 min end to end.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "prompt-caching"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/prompt-caching/lm.py', 'data/blog-2026-10-04/blog_train.txt']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The break-even, worked

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/prompt-caching/#the-break-even-worked)

In [ ]:
# When prompt caching pays, from the published multipliers (Anthropic's
# prompt-caching docs, October 2026): a cache write costs 1.25x the base
# input price with a 5-minute lifetime, 2x with a 1-hour lifetime; a cache
# read costs 0.1x (0.05x for Claude Opus 5.5, 0.025x for Claude Fable 5.1).
# A prefix of P tokens sent N times within the lifetime:
#   uncached: N * P;   cached: write * P + (N - 1) * read * P.
def cost(n, write, read=0.1):
    return write + (n - 1) * read                    # in units of P tokens at the base price

for name, write in (("5-minute", 1.25), ("1-hour", 2.0)):
    n = next(n for n in range(1, 100) if cost(n, write) < n)
    print(f"{name} cache (write {write}x, read 0.1x): cheaper from use {n} on; "
          f"at 10 uses it costs {cost(10, write) / 10:.0%} of uncached, at 100 uses {cost(100, write) / 100:.0%}")

# A 20,000-token system prompt for Claude Opus 5.5 (listed: $4 per million
# input tokens, $5 for 5-minute cache writes, $0.20 for cache reads), 1,000
# requests, each within 5 minutes of the previous one, so the cache stays
# warm and is written once.
P, price, N = 20_000, 4 / 1e6, 1_000
print(f"\n20k-token prefix, {N:,} requests: uncached ${N * P * price:,.2f}, "
      f"cached ${cost(N, 1.25, read=0.05) * P * price:,.2f}")

# What the provider holds for it: the prefix's KV cache, per cached copy.
for model, kv in (("Qwen2.5-1.5B", 28 * 1024), ("Qwen2.5-32B", 256 * 1024), ("DeepSeek-R1 (MLA)", 61 * 576 * 2)):
    print(f"KV cache for that 20k-token prefix, {model:18s}: {P * kv / 2**30:5.2f} GiB")

The post printed:

```text
5-minute cache (write 1.25x, read 0.1x): cheaper from use 2 on; at 10 uses it costs 22% of uncached, at 100 uses 11%
1-hour cache (write 2.0x, read 0.1x): cheaper from use 3 on; at 10 uses it costs 29% of uncached, at 100 uses 12%

20k-token prefix, 1,000 requests: uncached $80.00, cached $4.10
KV cache for that 20k-token prefix, Qwen2.5-1.5B      :  0.53 GiB
KV cache for that 20k-token prefix, Qwen2.5-32B       :  4.88 GiB
KV cache for that 20k-token prefix, DeepSeek-R1 (MLA) :  1.31 GiB
```

## What it saves in time

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/prompt-caching/#what-it-saves-in-time)

In [ ]:
import copy
import time
import numpy as np
import torch
from lm import model, tok, DEVICE
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"
gpu_sync = {"cuda": torch.cuda.synchronize, "mps": torch.mps.synchronize}.get(GPU, lambda: None)
gpu_empty_cache = {"cuda": torch.cuda.empty_cache, "mps": torch.mps.empty_cache}.get(GPU, lambda: None)

# What a cached prefix saves, measured with Qwen2.5-1.5B-Instruct on this
# laptop. A long shared prefix (instructions plus reference text, here text
# from this blog's posts) is followed by a short question.
#   cold:   the model processes prefix + question from scratch (prefill);
#   cached: the prefix's KV cache was computed once and kept; only the
#           question's tokens are processed, attending to it.
# Time to the first output token, median of 5, for prefixes of 1k to 8k.
m = model()
text = open("blog_train.txt").read()
question = tok("\n\nQuestion: what does this text say about the KV cache? Answer:", return_tensors="pt").input_ids.to(DEVICE)
cfg = m.config
kv_per_token = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * (cfg.hidden_size // cfg.num_attention_heads) * 2
print(f"KV cache per token: {kv_per_token / 1024:.0f} KiB (2 x {cfg.num_hidden_layers} layers x "
      f"{cfg.num_key_value_heads} KV heads x {cfg.hidden_size // cfg.num_attention_heads} x 2 bytes)")

@torch.no_grad()
def first_token(ids, past=None):
    gpu_sync(); t0 = time.perf_counter()
    out = m(ids, past_key_values=past, use_cache=True, logits_to_keep=1)   # logits for the last position only, as a server does
    out.logits[0, -1].argmax().item()                                 # forces the result
    gpu_sync()
    return time.perf_counter() - t0, out

print(f"{'prefix':>7s} {'cold':>9s} {'cached':>9s} {'saved':>6s} {'prefix KV held':>15s}")
for n in (1024, 2048, 4096, 8192):
    prefix = tok(text[:n * 6], return_tensors="pt").input_ids[:, :n].to(DEVICE)
    full = torch.cat([prefix, question], 1)
    first_token(full)                                                 # warm-up
    cold = np.median([first_token(full)[0] for _ in range(5)])
    _, pre = first_token(prefix)                                      # compute the prefix's cache once
    cached = np.median([first_token(question, copy.deepcopy(pre.past_key_values))[0] for _ in range(5)])
    print(f"{n:7d} {cold * 1e3:7.0f} ms {cached * 1e3:7.0f} ms {1 - cached / cold:6.0%} {n * kv_per_token / 2**20:11.0f} MiB")

The post printed:

```text
KV cache per token: 28 KiB (2 x 28 layers x 2 KV heads x 128 x 2 bytes)
 prefix      cold    cached  saved  prefix KV held
   1024     918 ms      52 ms    94%          28 MiB
   2048    1843 ms      60 ms    97%          56 MiB
   4096    3854 ms      77 ms    98%         112 MiB
   8192    8819 ms     112 ms    99%         224 MiB
```